# Projet HAR — Partie 1 : exploration des données

**Master 2 MALIA — Deep Learning — Université Lumière Lyon 2**
Binôme : DIOP Alioune, FALL Madjiguène

## Objectifs
Avant tout réseau de neurones, on applique l'étape 2 de la méthodologie du cours (*A recipe for training neural
networks* : **regarder les données**) :
1. connaître la taille du problème : nombre d'observations, de variables et de volontaires ;
2. vérifier la qualité des données : types, valeurs manquantes, doublons ;
3. examiner la **distribution des classes** ;
4. calculer des **statistiques descriptives** et vérifier si les variables sont déjà mises à l'échelle ;
5. produire quelques **visualisations** simples : histogrammes, boxplots, corrélations ;
6. décider des **prétraitements** (cours : *Data preparation*) et du rôle de la colonne `subject`.

## Plan
| Section | Contenu |
|---|---|
| 0 | Mise en place |
| 1 | Chargement et taille des données |
| 2 | Qualité des données |
| 3 | Distribution des classes |
| 4 | Statistiques descriptives |
| 5 | Visualisations |
| 6 | La colonne `subject` |
| 7 | Synthèse : les prétraitements retenus |

**Règle suivie dans tout le projet** (cours : *Training, validation and test sets*) : le fichier de test ne sert
à **aucune décision**. Dans ce notebook, on n'en lit que les colonnes `subject` et `Activity`, pour connaître sa
taille et sa composition ; toutes les statistiques des variables sont calculées sur `train.csv`.

---
## Partie 0 — Mise en place

In [ ]:
# Installation éventuelle (Google Colab ou machine personnelle) :
# !pip install pandas matplotlib seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(0)
pd.set_option("display.width", 140)

DATA_DIR = os.environ.get("HAR_DATA_DIR", "../data/HAR")         # dossier contenant train.csv et test.csv
FIG_DIR = os.environ.get("HAR_FIG_DIR", "../results/figures")    # figures réutilisées dans le rapport
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(name):
    # Enregistre la figure courante pour le rapport, puis l'affiche.
    plt.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight")
    plt.show()

CLASSES = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]

---
## Partie 1 — Chargement et taille des données

Chaque ligne est une **fenêtre** de 2,56 s de mesures (accéléromètre et gyroscope du smartphone porté à la ceinture),
résumée par 561 caractéristiques calculées à la main (moyennes, écarts-types, énergies, angles, etc.).
La colonne `subject` identifie le volontaire, la colonne `Activity` est la cible.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test_info = pd.read_csv(os.path.join(DATA_DIR, "test.csv"), usecols=["subject", "Activity"])

features = [c for c in df.columns if c not in ("subject", "Activity")]
print("train :", df.shape, "| colonnes de test lues :", list(df_test_info.columns))
df.iloc[:5, [0, 1, 2, 3, -2, -1]]

In [ ]:
taille = pd.DataFrame({
    "observations": [len(df), len(df_test_info)],
    "variables d'entrée": [len(features), len(features)],
    "volontaires": [df["subject"].nunique(), df_test_info["subject"].nunique()],
}, index=["train.csv", "test.csv"])
display(taille)

subj_train = sorted(df["subject"].unique())
subj_test = sorted(df_test_info["subject"].unique())
print("volontaires de train.csv :", subj_train)
print("volontaires de test.csv  :", subj_test)
print("volontaires communs      :", sorted(set(subj_train) & set(subj_test)))

**Question 1.1** — Combien d'observations, de variables et de volontaires contient chaque fichier ? Quelle est la
proportion train / test ? Les volontaires sont-ils les mêmes dans les deux fichiers, et qu'est-ce que cela implique
pour l'évaluation finale ?

*Réponse :*

---
## Partie 2 — Qualité des données

Cours (*Data preparation*, « Other steps ») : avant d'entraîner, on vérifie le type des variables, les **valeurs
manquantes** et les **doublons**.

In [ ]:
print("types des variables d'entrée :", df[features].dtypes.value_counts().to_dict())
print("valeurs manquantes (train)   :", df.isna().sum().sum())
print("lignes dupliquées (train)    :", df.duplicated().sum())

colonnes_identiques = df[features].T.duplicated()
print("colonnes identiques à une colonne précédente :", colonnes_identiques.sum())
if colonnes_identiques.sum() > 0:
    print("  exemples :", list(np.array(features)[colonnes_identiques.to_numpy()][:5]))

**Question 2.1** — Faut-il imputer des valeurs ou supprimer des lignes ? Que penser des colonnes dupliquées : gênent-
elles un réseau de neurones ?

*Réponse :*

---
## Partie 3 — Distribution des classes

La distribution des classes conditionne le choix de la **métrique** (cours : *Data preparation*, « Class imbalance :
weighted loss, resampling, appropriate metrics ») : avec des classes déséquilibrées, l'accuracy peut être trompeuse.

In [ ]:
distribution = pd.DataFrame({
    "train (effectif)": df["Activity"].value_counts(),
    "train (%)": 100 * df["Activity"].value_counts(normalize=True),
    "test (effectif)": df_test_info["Activity"].value_counts(),
    "test (%)": 100 * df_test_info["Activity"].value_counts(normalize=True),
}).reindex(CLASSES).round(1)
display(distribution)
print("rapport classe la plus fréquente / la moins fréquente (train) :",
      round(distribution["train (effectif)"].max() / distribution["train (effectif)"].min(), 2))

distribution[["train (%)", "test (%)"]].plot.bar(figsize=(8, 3.5))
plt.ylabel("% des fenêtres"); plt.title("Distribution des classes"); plt.xticks(rotation=30, ha="right")
save_fig("p1_distribution_classes")

Les volontaires n'ont pas tous enregistré le même nombre de fenêtres. Le tableau suivant donne, pour chaque
volontaire de `train.csv`, le nombre de fenêtres par activité.

In [ ]:
par_volontaire = pd.crosstab(df["subject"], df["Activity"])[CLASSES]
plt.figure(figsize=(9, 6))
sns.heatmap(par_volontaire, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Nombre de fenêtres par volontaire et par activité (train)"); plt.xlabel(""); plt.ylabel("volontaire")
save_fig("p1_fenetres_par_volontaire")
print("fenêtres par volontaire : min", par_volontaire.sum(axis=1).min(), "| max", par_volontaire.sum(axis=1).max())

**Question 3.1** — Les classes sont-elles équilibrées ? La distribution est-elle la même dans `train.csv` et
`test.csv` ? Quelle conséquence pour le choix entre accuracy et F1-score (à reprendre en partie 2.2 du projet) ?

*Réponse :*

---
## Partie 4 — Statistiques descriptives

On calcule, pour chaque variable, la moyenne, l'écart-type, le minimum et le maximum (sur `train.csv` uniquement).

In [ ]:
stats = df[features].describe().T[["mean", "std", "min", "max"]]
display(stats.head(10).round(3))

print(f"minimum global : {stats['min'].min():.3f} | maximum global : {stats['max'].max():.3f}")
print(f"variables comprises dans [-1, 1] : {((stats['min'] >= -1) & (stats['max'] <= 1)).mean():.1%}")
print(f"moyennes     : de {stats['mean'].min():.3f} à {stats['mean'].max():.3f}")
print(f"écarts-types : de {stats['std'].min():.3f} à {stats['std'].max():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(stats["mean"], bins=50); axes[0].set_title("Moyenne des 561 variables"); axes[0].set_xlabel("moyenne")
axes[1].hist(stats["std"], bins=50, color="C1"); axes[1].set_title("Écart-type des 561 variables"); axes[1].set_xlabel("écart-type")
plt.tight_layout(); save_fig("p1_moyennes_ecarts_types")

**Question 4.1** — Les variables sont-elles déjà mises à l'échelle ? Leurs moyennes et écarts-types sont-ils
comparables ? Faut-il quand même les standardiser, $x_j \leftarrow (x_j-\mu_j)/\sigma_j$ ? Rappelez pourquoi
(cours : *Data preparation*, « features on very different scales ⇒ ill-conditioned loss, saturated neurons ») et sur
quel ensemble $\mu_j$ et $\sigma_j$ doivent être calculés.

*Réponse :*

---
## Partie 5 — Visualisations

### 5.1 Histogrammes et boxplots par activité
On choisit quelques variables faciles à interpréter physiquement :
- `tBodyAccMag-mean()` : intensité moyenne de l'accélération due au mouvement du corps ;
- `tBodyAcc-std()-X` : variabilité de cette accélération selon l'axe X ;
- `tGravityAcc-mean()-X` : composante de la gravité selon l'axe X, liée à l'**orientation** du téléphone ;
- `angle(X,gravityMean)` : angle entre l'axe X du téléphone et la gravité ;
- `fBodyAccMag-mean()` : même intensité, dans le domaine fréquentiel ;
- `tBodyGyro-std()-X` : variabilité de la vitesse de rotation.

In [ ]:
variables = ["tBodyAccMag-mean()", "tBodyAcc-std()-X", "tGravityAcc-mean()-X",
             "angle(X,gravityMean)", "fBodyAccMag-mean()", "tBodyGyro-std()-X"]
variables = [v for v in variables if v in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, v in zip(axes.ravel(), variables):
    for k in CLASSES:
        ax.hist(df.loc[df["Activity"] == k, v], bins=40, alpha=0.5, density=True, label=k)
    ax.set_title(v)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); save_fig("p1_histogrammes")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, v in zip(axes.ravel(), variables):
    sns.boxplot(data=df, x="Activity", y=v, order=CLASSES, ax=ax, fliersize=1)
    ax.set_xlabel(""); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); save_fig("p1_boxplots")

**Question 5.1** — Quelles variables séparent les activités **dynamiques** (marche, montée, descente) des activités
**statiques** (assis, debout, allongé) ? Laquelle isole l'activité LAYING, et pourquoi (pensez à l'orientation du
téléphone par rapport à la gravité) ? Quelles activités semblent les plus difficiles à distinguer ?

*Réponse :*

### 5.2 Corrélations entre variables
Beaucoup de variables sont calculées à partir des mêmes signaux (même capteur, même axe, domaine temporel ou
fréquentiel) : on s'attend à de fortes corrélations.

In [ ]:
corr = df[features].corr()
plt.figure(figsize=(8, 7))
sns.heatmap(corr.iloc[:60, :60], cmap="coolwarm", center=0, vmin=-1, vmax=1, xticklabels=False, yticklabels=False)
plt.title("Corrélations entre les 60 premières variables")
save_fig("p1_correlations")

triangle = corr.abs().to_numpy()[np.triu_indices(len(features), k=1)]     # chaque paire comptée une fois
for seuil in [0.8, 0.9, 0.95]:
    print(f"paires de variables avec |corrélation| > {seuil} : {(triangle > seuil).sum()} "
          f"sur {len(triangle)} ({(triangle > seuil).mean():.1%})")

**Question 5.2** — Les variables sont-elles redondantes ? Faut-il supprimer les variables très corrélées avant
d'entraîner un MLP ? (La redondance n'empêche pas l'apprentissage : la première couche apprend elle-même des
combinaisons des entrées. On peut en discuter l'intérêt en temps de calcul et en interprétabilité.)

*Réponse :*

---
## Partie 6 — La colonne `subject` doit-elle servir d'entrée ?

`subject` est un **identifiant** (un numéro de volontaire), pas une mesure. Regardons tout de même si les mesures
dépendent du volontaire : pour une même activité, les fenêtres d'un volontaire se ressemblent-elles plus entre
elles qu'avec celles des autres ?

In [ ]:
v = "tBodyAccMag-mean()" if "tBodyAccMag-mean()" in df.columns else features[0]
marche = df[df["Activity"] == "WALKING"]
plt.figure(figsize=(12, 3.5))
sns.boxplot(data=marche, x="subject", y=v, fliersize=1, color="C0")
plt.title(f"{v} pendant la marche, par volontaire (train)"); plt.xlabel("volontaire")
save_fig("p1_variable_par_volontaire")

moyennes_vol = marche.groupby("subject")[v].mean()
print(f"écart-type de {v} entre les fenêtres de marche : {marche[v].std():.3f}")
print(f"écart-type des moyennes par volontaire        : {moyennes_vol.std():.3f}")

**Question 6.1** — Les mesures dépendent-elles du volontaire ? La colonne `subject` doit-elle servir d'**entrée** au
modèle ? (Rappel : les volontaires du test sont absents de `train.csv`.) À quoi servira-t-elle en revanche dans la
suite du projet (partie 1.3 : construction de la validation) ?

*Réponse :*

---
## Partie 7 — Synthèse : les prétraitements retenus

**Exercice 7.1** — Complétez le tableau à partir de **vos** observations (il servira pour la section « Données » du
rapport).

| Étape | Décision | Justification (observation de ce notebook) |
|---|---|---|
| Valeurs manquantes, doublons | | |
| Mise à l'échelle des entrées | | |
| Statistiques de normalisation calculées sur… | | |
| Encodage de la cible `Activity` | | |
| Colonne `subject` | | |
| Variables corrélées / dupliquées | | |
| Métrique principale | | |